# 02 · Playlist Analysis

You don't really use "Liked Songs", so for your profile the truest picture of
your taste is **the songs you've put into your own playlists**. We treat that
as your library.

Source: `data/api/playlist_tracks.csv` (every track from every accessible
playlist). We filter to the playlists **you own**, then look at:

1. Most common artists
2. Song-length distribution
3. Release-year distribution
4. Decade breakdown
5. Biggest playlists
6. Artists that span the most playlists ("breadth")
7. Songs you've added to the most playlists ("repeats")
8. When you added songs over time

Still no genres/audio features (Spotify restricts those for new apps) — all
behavioral + metadata.


## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid")
GREEN = "#1DB954"


## Load & scope to your own playlists

`playlist_tracks.csv` includes tracks from playlists owned by other people too
(ones you follow). We keep only playlists **you** own. Change `MY_NAME` if your
Spotify display name is different.


In [ ]:
from pathlib import Path

# Find a project file no matter which folder the notebook is launched from
# (browser Jupyter runs from notebooks/, VS Code runs from the project root).
def project_file(rel):
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / rel
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {rel} starting from {Path.cwd()}")

raw = pd.read_csv(project_file("data/api/playlist_tracks.csv"))

MY_NAME = "gene"   # your Spotify display name (the 'playlist_owner' value)
mine = raw[raw["playlist_owner"] == MY_NAME].copy()

# Playlists to exclude from the analysis (single-artist dumps, junk, etc.).
BANNED_PLAYLISTS = [
    "background music and meme songs",
    "turi ip ip ip 😂🔥⁉️🚨😈😹🆒🔝",
    "every teardrop is a waterfall",
    "how to disappear completely",
    "john",
    "pink floyd",
    "smiths",
    "metaphorical music",
    "band i trust",
    "the paper kites/kings of convenience",
    "beats",
    "bad playlist",
    "best spotify mix party transitions",
]
mine = mine[~mine["playlist"].isin(BANNED_PLAYLISTS)].copy()

# Derived columns.
mine["added_at"] = pd.to_datetime(mine["added_at"], utc=True, errors="coerce")
mine["duration_min"] = mine["duration_ms"] / 60000
mine["release_year"] = pd.to_numeric(mine["release_date"].astype(str).str[:4],
                                     errors="coerce")

# Drop stub rows: local files / unavailable tracks with no real metadata
# (missing name or zero duration).
mine = mine[mine["track"].notna() & (mine["duration_ms"] > 0)].copy()

print(f"Playlist track entries (own playlists): {len(mine)}")
print(f"Unique songs:                           {mine['track_uri'].nunique()}")
print(f"Playlists:                              {mine['playlist'].nunique()}")
print(f"Unique artists:                         {mine['artist'].nunique()}")
mine.head()


## 1. Most common artists

Counted over **every playlist appearance** (`mine`), so a song you've put in
several playlists contributes each time. This rewards artists you spread across
your playlists.

In [ ]:
top_artists = mine["artist"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=top_artists.values, y=top_artists.index, color=GREEN, ax=ax)
ax.set_title("Most common artists across your playlists")
ax.set_xlabel("Number of songs")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 2. Song-length distribution

How long are the songs you save?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
mine["duration_min"].plot(kind="hist", bins=30, color=GREEN,
                             edgecolor="white", ax=ax)
ax.axvline(mine["duration_min"].median(), color="black",
           linestyle="--", label=f"median {mine['duration_min'].median():.1f} min")
ax.set_title("Song length distribution")
ax.set_xlabel("Minutes")
ax.set_ylabel("Number of songs")
ax.legend()
plt.tight_layout()
plt.show()

longest = mine.loc[mine["duration_min"].idxmax()]
shortest = mine.loc[mine["duration_min"].idxmin()]
print(f"Longest:  {longest['track']} ({longest['duration_min']:.1f} min)")
print(f"Shortest: {shortest['track']} ({shortest['duration_min']:.1f} min)")


## 3. Release-year distribution

Are you into new music or older catalog?

In [ ]:
# Drop missing years and any obviously-bad ones (e.g. blank release dates).
years = mine["release_year"].dropna()
years = years[years >= 1900]

lo, hi = int(years.min()), int(years.max())

fig, ax = plt.subplots(figsize=(11, 4))
# range=(lo, hi) makes the bars span from your oldest to newest song only.
ax.hist(years, bins=30, range=(lo, hi), color=GREEN, edgecolor="white")
ax.set_xlim(lo, hi)                      # start the axis at your lowest year
ax.set_title("Release years of your songs")
ax.set_xlabel("Release year")
ax.set_ylabel("Number of songs")
plt.tight_layout()
plt.show()

# Look up which song is the oldest and newest.
valid = mine[mine["release_year"] >= 1900]
oldest = valid.loc[valid["release_year"].idxmin()]
newest = valid.loc[valid["release_year"].idxmax()]

print(f"Oldest: {lo} — {oldest['track']} by {oldest['artist']}")
print(f"Newest: {hi} — {newest['track']} by {newest['artist']}")
print(f"Median: {int(years.median())}")

## 4. Decade breakdown

Same information as above, grouped into decades — often easier to read.


In [ ]:
decade = (mine["release_year"] // 10 * 10).dropna().astype(int)
by_decade = decade.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(x=[f"{d}s" for d in by_decade.index], y=by_decade.values,
            color=GREEN, ax=ax)
ax.set_title("Songs by decade")
ax.set_xlabel("")
ax.set_ylabel("Number of songs")
plt.tight_layout()
plt.show()


## 5. Biggest playlists

Which of your playlists hold the most tracks?


In [ ]:
sizes = mine["playlist"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=sizes.values, y=sizes.index, color=GREEN, ax=ax)
ax.set_title("Your biggest playlists")
ax.set_xlabel("Number of tracks")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 6. Artists with the widest reach

Not who has the most *songs*, but who shows up across the most *different*
playlists — the artists woven through your whole listening life.


In [ ]:
breadth = (mine.groupby("artist")["playlist"].nunique()
           .sort_values(ascending=False).head(15))

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=breadth.values, y=breadth.index, color=GREEN, ax=ax)
ax.set_title("Artists appearing in the most playlists")
ax.set_xlabel("Number of playlists")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 7. Songs you keep reaching for

Tracks you've added to several different playlists — your personal anthems.


In [ ]:
repeats = (mine.groupby(["track", "artist"])["playlist"].nunique()
           .sort_values(ascending=False))
repeats = repeats[repeats > 1].head(15)

if len(repeats):
    for (track, artist), n in repeats.items():
        print(f"  {n}x  {track} — {artist}")
else:
    print("No song appears in more than one of your playlists.")


## 8. When did you add songs?

Using each track's `added_at`, we can see your playlist-building activity over
time — bursts of curation vs. quiet stretches.


In [ ]:
monthly = mine.dropna(subset=["added_at"]).copy()
monthly["month"] = monthly["added_at"].dt.strftime("%Y-%m")
counts = monthly.groupby("month").size()

if len(counts):
    all_months = pd.period_range(start=counts.index.min(),
                                 end=counts.index.max(), freq="M").astype(str)
    counts = counts.reindex(all_months, fill_value=0)

    fig, ax = plt.subplots(figsize=(12, 4))
    counts.plot(kind="bar", ax=ax, color=GREEN)
    ax.set_title("Songs added to playlists per month")
    ax.set_xlabel("Month")
    ax.set_ylabel("Songs added")
    # Thin the x labels if there are many months.
    step = max(1, len(counts) // 24)
    for i, lbl in enumerate(ax.get_xticklabels()):
        lbl.set_visible(i % step == 0)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No 'added_at' dates available.")


## Wrap-up & ideas to extend

You now have a real portrait of your taste from your own playlists.

**Try yourself:**
- Filter to one playlist (`mine[mine['playlist'] == 'night drive']`) and rerun a chart.
- Compare two playlists' median release years.

**Bigger next steps (later):**
- Join by **ISRC** to an outside dataset to finally add genres + audio features,
  unlocking mood charts and k-means clustering.
- Bring in your full **streaming-history export** for actual play counts and
  time-of-day patterns (this notebook is about what you *saved*, not what you
  *played*).
